In [2]:

# Imports
import pandas as pd
import torch
import numpy as np
import json
import os
import random
from collections import defaultdict



### Extract initial list of singer and file paths from directory
Map singer <=> idXXXXX to adhere to Kaldi format

In [13]:
# Directory containing the files
directory = os.path.expanduser('~/gcs-mount/metamidi_demucs_stems')

# Initialize a dictionary to store file paths per artist
artist_file_paths = defaultdict(list)

# Iterate over files in the directory
for filename in os.listdir(directory):
    if filename.endswith('.wav'):
        # Extract the artist name from the filename
        parts = filename.split('-')
        if len(parts) > 2:
            artist_name = parts[-2]
            # Store the full path to the file
            full_path = os.path.join(directory, filename)
            artist_file_paths[artist_name].append(full_path)

# Store artist names in a set to ensure uniqueness
unique_artists = set(artist_file_paths.keys())
print(f"Number of unique artists: {len(unique_artists)}")



# Map artist names to singer_ids
def map_artist_to_singer_id(artists):
    """
    Maps each artist name to a unique singer_id in the form "idXXXXX".
    """
    singer_id_dict = {}
    base_id = 10000  # Start with "id1XXXX"
    for i, artist in enumerate(sorted(artists)):
        # Calculate the singer_id
        singer_id = f"id{(i // base_id) + 1}{i % base_id:04d}"
        singer_id_dict[artist] = singer_id
    return singer_id_dict

singer_id_mapping = map_artist_to_singer_id(unique_artists)

# Create the JSON structure
singer_data = {}
for artist, singer_id in singer_id_mapping.items():
    singer_data[singer_id] = {
        "artist_name": artist,
        "audio_paths": artist_file_paths.get(artist, [])
    }

# Save the JSON structure to a file
json_file_path = "singer_data.json"
with open(json_file_path, 'w') as json_file:
    json.dump(singer_data, json_file, indent=2)

print(f"Singer data saved to {json_file_path}")

Number of unique artists: 14100
Singer data saved to singer_data.json


We want to do a 10% test split but only using singers with 2-4 songs in the dataset.

In [17]:
import random

# From singer_data, make a list of singers with 2-4 songs
two_to_four_files_artists_set = set(artist for artist in unique_artists if 2 <= len(artist_file_paths[artist]) <= 4)

# Calculate the number of test singers if we want to do a 10% test split
num_test_singers = int(len(unique_artists) * 0.1)

# Calculate the ratio of singers with 2-4 files we need to assign to the test set
ratio_test_singers = num_test_singers / len(two_to_four_files_artists_set)

# Random seed for reproducibility
random.seed(42)

# Convert the set to a list before sampling
two_to_four_files_artists_list = list(two_to_four_files_artists_set)

# Randomly assign ratio_test_singers in singers with 2-4 files to the test set
test_singers = random.sample(two_to_four_files_artists_list, num_test_singers)

# Create a new set of test singers
test_singers_set = set(test_singers)

print(f"Number of test singers: {len(test_singers_set)}")

# Assign the rest of the singers in unique_artists who are not in test_singers_set to the train set
train_singers_set = unique_artists - test_singers_set

# For each artist in test_singers_set, look for the corresponding singer_id in singer_data and assign a new key-value pair in singer_data with key: "is_test" and value: 1
for artist in test_singers_set:
    singer_id = singer_id_mapping[artist]
    singer_data[singer_id]["is_test"] = 1

# For each artist in train_singers_set, look for the corresponding singer_id in singer_data and assign a new key-value pair in singer_data with key: "is_test" and value: 0
for artist in train_singers_set:
    singer_id = singer_id_mapping[artist]
    singer_data[singer_id]["is_test"] = 0

# Save the updated singer_data to the same json file
with open(json_file_path, 'w') as json_file:
    json.dump(singer_data, json_file, indent=2)

print(f"Singer data saved to {json_file_path}")

Number of test singers: 1410
Singer data saved to singer_data.json


In [7]:
import json

# Define the singer_data.json file path
file_path = '/home/aik2/sc-rawnet3/datasets/metamidi/singer_data.json'

# First, load the current data
with open(file_path, 'r') as f:
    singer_data = json.load(f)

# Find the singer with artist name "Trad"
trad_singer_id = None
for singer_id, singer_info in singer_data.items():
    if singer_info.get('artist_name') == 'Trad':
        trad_singer_id = singer_id
        break

# Remove the singer if found
if trad_singer_id:
    del singer_data[trad_singer_id]
    print(f"Found and will remove singer ID: {trad_singer_id}")
else:
    print("No singer with artist name 'Trad' found.")

# Write the updated data back to the file
with open(file_path, 'w') as f:
    json.dump(singer_data, f, indent=2)
    f.flush()  # Ensure data is written to disk

print("File has been updated and saved.")

No singer with artist name 'Trad' found.
File has been updated and saved.


In [1]:
import json
import os
import shutil
from tempfile import NamedTemporaryFile

# Define the singer_data.json file path
file_path = '/home/aik2/sc-rawnet3/datasets/metamidi/singer_data.json'

# Create a backup first
backup_path = file_path + '.backup'
print(f"Creating backup at {backup_path}")
shutil.copy2(file_path, backup_path)

try:
    # Load the data
    with open(file_path, 'r') as f:
        singer_data = json.load(f)
    
    # Find singers with artist name "Traditional"
    traditional_singers = []
    for singer_id, singer_info in singer_data.items():
        if singer_info.get('artist_name') == 'Traditional':
            traditional_singers.append(singer_id)
    
    # Remove the singers if found
    if traditional_singers:
        for singer_id in traditional_singers:
            del singer_data[singer_id]
        print(f"Found and removed {len(traditional_singers)} singers with name 'Traditional': {', '.join(traditional_singers)}")
    else:
        print("No singers with artist name 'Traditional' found.")
    
    # Write to a temporary file first
    with NamedTemporaryFile('w', delete=False) as temp_file:
        temp_path = temp_file.name
        json.dump(singer_data, temp_file, indent=2)
        temp_file.flush()
        os.fsync(temp_file.fileno())  # Ensure data is written to disk
    
    # Replace the original file with the temporary file
    shutil.move(temp_path, file_path)
    
    print("File has been safely updated and saved.")
    print(f"If needed, you can restore from the backup at {backup_path}")
    
except Exception as e:
    print(f"An error occurred: {str(e)}")
    print(f"The original file is backed up at {backup_path}")

Creating backup at /home/aik2/sc-rawnet3/datasets/metamidi/singer_data.json.backup
Found and removed 1 singers with name 'Traditional': id23231
File has been safely updated and saved.
If needed, you can restore from the backup at /home/aik2/sc-rawnet3/datasets/metamidi/singer_data.json.backup


## Downsample and redirect
Done using a separate python script, downsample_redicrect.py
Run that script before continuing onto the next steps

## Update singer_data 
to include the new 16k paths


In [19]:
import json
import os
from pathlib import Path

# Load the singer data from the JSON file
with open('/home/aik2/sc-rawnet3/datasets/metamidi/singer_data.json', 'r') as f:
    singer_data = json.load(f)

# Redirect save path
redirect_path = '/home/aik2/sc-rawnet3/datasets/metamidi/audio_16k'

# Update singer_data with new 16k paths
for singer_id, singer_info in singer_data.items():
    # Determine if the singer is in the train or test set
    set_type = 'test' if singer_info.get("is_test", 0) == 1 else 'train'
    
    # Access audio paths
    audio_paths = singer_info['audio_paths']
    path_16k_list = []
    
    for audio_path in audio_paths:
        # Extract the folder name by removing the "-vocals.wav" suffix
        filename = Path(audio_path).name
        folder_name = filename.rsplit('-vocals.wav', 1)[0]
        
        # Construct the new path for the 16k audio file
        target_dir = os.path.join(redirect_path, set_type, singer_id, folder_name)
        new_path = os.path.join(target_dir, '00001.wav')
        
        # Add the new path to the list
        path_16k_list.append(new_path)
    
    # Add the new paths to the singer_data under the key "path_16k"
    singer_info['path_16k'] = path_16k_list

# Save the updated singer_data to the JSON file
with open('/home/aik2/sc-rawnet3/datasets/metamidi/singer_data.json', 'w') as f:
    json.dump(singer_data, f, indent=2)

print("Updated singer_data.json with new 16k paths.")

Updated singer_data.json with new 16k paths.


### Now create comparison pairs for the test set

In [1]:
import json
import os
import random
from itertools import combinations
import pandas as pd

# Load the singer_data.json
with open('/home/aik2/sc-rawnet3/datasets/metamidi/singer_data.json', 'r') as f:
    singer_data = json.load(f)

# Make a list of all 16k audio file paths in the test set
all_files = []
test_singers = []

# First, identify all test singers and collect their file paths
for singer_id, singer_info in singer_data.items():
    if singer_info.get("is_test", 0) == 1:
        test_singers.append(singer_id)
        all_files.extend(singer_info.get("path_16k", []))

print(f"Found {len(test_singers)} test singers with a total of {len(all_files)} audio files.")

# Initialize list to store all pairs
all_pairs = []

# Iterate over all singers in the test set
for singer_id in test_singers:
    singer_files = singer_data[singer_id].get("path_16k", [])
    num_files = len(singer_files)
    
    if num_files < 2:
        print(f"Skipping singer {singer_id} with fewer than 2 files.")
        continue
    
    # Create true pairs (for files from the same singer)
    true_pairs = list(combinations(singer_files, 2))
    # print(f"Created {len(true_pairs)} true pairs for singer {singer_id}")
    
    # Create a list of files from other singers
    other_singers_files = [f for f in all_files if f not in singer_files]
    
    # Create false pairs (pair each file with a file from a different singer)
    false_pairs = []
    
    # First, iterate through each file in singer_files once
    for file1 in singer_files:
        # Select a random file from other singers
        file2 = random.choice(other_singers_files)
        
        false_pairs.append((file1, file2))
        
        # Try to avoid reusing the same file immediately if possible
        if len(other_singers_files) > 1:
            other_singers_files.remove(file2)
        
        # If we have enough false pairs, break
        if len(false_pairs) >= len(true_pairs):
            break
    
    # If we still need more false pairs, create them with random selection from singer_files
    while len(false_pairs) < len(true_pairs):
        file1 = random.choice(singer_files)
        file2 = random.choice(other_singers_files)
        
        false_pairs.append((file1, file2))
        
        # Try to avoid reusing the same file immediately if possible
        if len(other_singers_files) > 1:
            other_singers_files.remove(file2)
    
    # Add to all_pairs, alternating true and false pairs
    for i in range(len(true_pairs)):
        # Add true pair
        all_pairs.append([1, true_pairs[i][0], true_pairs[i][1]])
        
        # Add false pair
        all_pairs.append([0, false_pairs[i][0], false_pairs[i][1]])

# Convert to DataFrame
comparison_df = pd.DataFrame(all_pairs, columns=["is_same_singer", "file_path_1", "file_path_2"])

# Save to comparison_pairs.txt
comparison_df.to_csv("/home/aik2/sc-rawnet3/datasets/metamidi/comparison_pairs.txt", 
                      sep="\t", index=False, header=False)

print(f"Created {len(comparison_df)} comparison pairs.")
print(f"True pairs: {len(comparison_df[comparison_df['is_same_singer'] == 1])}")
print(f"False pairs: {len(comparison_df[comparison_df['is_same_singer'] == 0])}")
print(f"Saved to comparison_pairs.txt")

Found 1410 test singers with a total of 3734 audio files.
Created 7008 comparison pairs.
True pairs: 3504
False pairs: 3504
Saved to comparison_pairs.txt
